# Практика 2. Исследуем предобученный BERT



## 0. Установка и импорты



In [ ]:
# %pip install -q "transformers>=4.44,<5" matplotlib

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer, pipeline

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "bert-base-multilingual-cased"
print("device:", device)


## 1. Быстрый инференс через pipeline

`pipeline` удобен для проверки идеи, но прячет токенизацию, batching и post-processing. Сначала посмотрим на результат, затем воспроизведём его вручную.


In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model=MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
)
fill_mask("Студент открыл [MASK] и начал писать код.", top_k=5)


## 2. WordPiece и входы модели

BERT получает не строки, а числовые tensors. Посмотрим, как слова разбиваются на subword-токены, и сравним padding справа в двух предложениях.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texts = [
    "Трансформеры учитывают контекст.",
    "Необычное псевдослово трансформеризация разбивается на части.",
]
batch = tokenizer(texts, padding=True, return_tensors="pt")

for key, value in batch.items():
    print(f"{key:16s}", tuple(value.shape), value[0].tolist())
print("\ntokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][1]))


## 3. Hidden states

`AutoModel` возвращает контекстные представления всех токенов. При `output_hidden_states=True` получаем embedding output и выход каждого encoder-слоя.


In [ ]:
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
encoded = tokenizer("Кошка сидит на тёплом окне.", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = encoder(**encoded, output_hidden_states=True, output_attentions=True)

print("last_hidden_state:", outputs.last_hidden_state.shape)
print("hidden states:", len(outputs.hidden_states))
print("attentions:", len(outputs.attentions), outputs.attentions[0].shape)


Для BERT-base ожидаем 13 hidden-state tensors: embeddings + 12 encoder-слоёв. Attention каждого слоя имеет форму `(B, H, T, T)`.


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
# TODO: извлеките вектор [CLS] из каждого hidden state, сложите через torch.stack
# и посчитайте cosine similarity между соседними слоями.
cls_by_layer = None


## 4. Визуализация attention

Визуализация полезна для диагностики, но attention weight нельзя автоматически считать полным объяснением решения модели. Одна голова может смотреть на разделители, другая — на близкие токены, третья — почти равномерно.


In [ ]:
def plot_attention(attention, tokens, layer=0, head=0):
    # attention[layer]: (B,H,T,T); выбираем batch=0 и одну голову.
    matrix = attention[layer][0, head].detach().cpu()
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap="viridis")
    ax.set_xticks(range(len(tokens)), tokens, rotation=60, ha="right")
    ax.set_yticks(range(len(tokens)), tokens)
    # Строка — токен-запрос (query), столбец — токен-источник (key).
    ax.set_xlabel("key tokens")
    ax.set_ylabel("query tokens")
    ax.set_title(f"layer={layer}, head={head}")
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()

plot_attention(outputs.attentions, tokens, layer=0, head=0)
plot_attention(outputs.attentions, tokens, layer=11, head=7)


## 5. Masked Language Modeling вручную

Воспроизведите результат `pipeline("fill-mask")` через `AutoModelForMaskedLM`: найдите индекс `[MASK]`, возьмите logits этой позиции, примените softmax и выведите top-10 токенов.


In [ ]:
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
sentence = "Сегодня на улице очень [MASK], поэтому возьми куртку."
mlm_inputs = tokenizer(sentence, return_tensors="pt").to(device)
# nonzero возвращает пары (batch_index, token_index).
mask_positions = (mlm_inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)
assert len(mask_positions) == 1, "Ожидается ровно один [MASK]"

with torch.no_grad():
    # logits имеют форму (B,T,V), хотя нас интересует только одна позиция T.
    logits = mlm_model(**mlm_inputs).logits

batch_index, token_index = mask_positions[0]
probabilities = logits[batch_index, token_index].softmax(-1)
# topk возвращает отдельно значения вероятностей и id токенов.
top = probabilities.topk(10)
for token_id, probability in zip(top.indices, top.values):
    token = tokenizer.decode([token_id])
    print(f"{token!r:18s} {probability.item():.3f}")


## 6. Контекстные представления

Напишите функцию, которая возвращает средний hidden vector subword-токенов заданного слова. Используйте `return_offsets_mapping=True`. Сравните cosine similarity слова «лук» в контекстах про оружие и овощ.


In [ ]:
def phrase_vector(text, phrase):
    # Сначала находим символьные границы слова в исходной строке.
    start = text.lower().index(phrase.lower())
    end = start + len(phrase)
    encoded = tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
    offsets = encoded.pop("offset_mapping")[0]
    # Выбираем все subword-токены, чьи интервалы пересекаются с искомым словом.
    selected = [i for i, (a, b) in enumerate(offsets.tolist()) if a < end and b > start]
    encoded = encoded.to(device)
    with torch.no_grad():
        hidden = encoder(**encoded).last_hidden_state[0]
    # Слово может состоять из нескольких WordPiece; сводим их к одному вектору.
    return hidden[selected].mean(0)

contexts = [
    "Лучник натянул лук и прицелился.",
    "Повар мелко нарезал зелёный лук.",
    "Спортсмен взял лук перед соревнованием.",
]

vectors = [phrase_vector(text, "лук") for text in contexts]

for i in range(len(contexts)):
    for j in range(i + 1, len(contexts)):
        similarity = F.cosine_similarity(vectors[i], vectors[j], dim=0).item()
        print(f"{i} vs {j}: {similarity:.3f}")